# Conciliação de Benefícios
**Instituto Atlântico · Departamento Pessoal**

---

Este notebook realiza a conciliação mensal entre faturas de fornecedores e os dados de referência internos do DP.

### Fornecedores suportados
| Fornecedor | Identificação automática pelo nome do arquivo |
|---|---|
| Bradesco Dental | nome contém `bradesco` + `dental` |

> Para adicionar um novo fornecedor, veja as instruções na **Célula 2 — Registro de parsers**.

---

### Como usar
1. Execute a **Célula 1** para instalar dependências *(uma vez por sessão)*
2. Execute a **Célula 2** para carregar os parsers
3. Execute a **Célula 3** para selecionar e confirmar os arquivos
4. Execute a **Célula 4** para processar e visualizar o resultado

In [ ]:
#@title Célula 1 · Preparação do ambiente { display-mode: "form" }
#@markdown Execute uma única vez por sessão.

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "pandas"])

import io
import pandas as pd
import openpyxl
from IPython.display import display, HTML
from google.colab import files
import ipywidgets as widgets

# Estado global compartilhado entre células
estado = {
    "uploaded":           {},   # {nome_arquivo: bytes}
    "arquivo_fatura":     None,
    "arquivo_referencia": None,
    "fornecedor":         None,
}

print("✅ Ambiente pronto.")

In [ ]:
#@title Célula 2 · Registro de parsers { display-mode: "form" }
#@markdown Cada fornecedor tem uma função `parse_fatura_*` registrada aqui.
#@markdown Para adicionar um novo fornecedor:
#@markdown 1. Crie a função `parse_fatura_<nome>(conteudo)` seguindo o padrão abaixo
#@markdown 2. Adicione uma entrada em `PARSERS` com a chave de identificação e a função
#@markdown 3. Adicione a entrada na tabela de fornecedores suportados no cabeçalho

TOLERANCIA = 0.05  # diferença máxima em R$ para considerar OK (arredondamento)


# ════════════════════════════════════════════════════════════════════════
# PARSER: Bradesco Dental
# Agrega desconto por matrícula titular (titular + dependentes)
# ════════════════════════════════════════════════════════════════════════

def parse_fatura_bradesco_dental(conteudo):
    wb = openpyxl.load_workbook(io.BytesIO(conteudo), data_only=True)
    ws = wb["bradesco"]

    # Localizar linha de cabeçalho pela presença de 'Matricula Titular'
    header_row = None
    for i, row in enumerate(ws.iter_rows(values_only=True)):
        if any(str(v).strip().lower() == "matricula titular" for v in row if v):
            header_row = i
            break
    if header_row is None:
        raise ValueError("Coluna 'Matricula Titular' não encontrada na aba 'bradesco'.")

    headers = list(ws.iter_rows(values_only=True))[header_row]
    idx = {}
    for i, h in enumerate(headers):
        h_norm = str(h).strip().lower() if h else ""
        if h_norm == "matricula titular":      idx["mat"]     = i
        elif h_norm == "desconto colaborador": idx["desconto"] = i
        elif h_norm in ["x", "custo"] and "custo" not in idx:
                                               idx["custo"]   = i

    registros = []
    for row in list(ws.iter_rows(values_only=True))[header_row + 2:]:
        mat  = row[idx["mat"]]     if "mat"     in idx else None
        desc = row[idx["desconto"]] if "desconto" in idx else None
        custo = row[idx["custo"]]  if "custo"   in idx else None
        if mat and isinstance(mat, (int, float)) and desc:
            registros.append({
                "matricula":      int(mat),
                "desconto_fatura": float(desc),
                "custo_fatura":    float(custo) if custo else 0.0,
            })

    df = pd.DataFrame(registros)
    return df.groupby("matricula").agg(
        desconto_fatura=("desconto_fatura", "sum"),
        custo_fatura=("custo_fatura", "sum"),
        qtd_vidas=("matricula", "count"),
    ).reset_index()


def parse_referencia_interna(conteudo):
    """
    Lê a referência interna do DP (aba 'total') e retorna os valores
    esperados de Bradesco Dental por colaborador.
    """
    wb = openpyxl.load_workbook(io.BytesIO(conteudo), data_only=True)
    ws = wb["total"]
    rows = list(ws.iter_rows(values_only=True))

    idx_mat = idx_nome = idx_fatura = idx_desconto = idx_custo = None
    idx_bd_start = header1_row = header2_row = None

    for i, row in enumerate(rows):
        vals = [str(v).strip().lower() if v else "" for v in row]
        if "mat" in vals and "nome" in vals:
            header1_row = i
            idx_mat  = next(j for j, v in enumerate(vals) if v == "mat")
            idx_nome = next(j for j, v in enumerate(vals) if v == "nome")
            for row2 in [rows[i], rows[i+1] if i+1 < len(rows) else []]:
                for j, v in enumerate(row2):
                    if v and "bradesco dental" in str(v).lower():
                        idx_bd_start = j
                        header2_row  = i + 1
                        break
            break

    if header2_row:
        subheaders = rows[header2_row]
        for j, v in enumerate(subheaders):
            if v and "fatura" in str(v).lower() and idx_bd_start and j >= idx_bd_start - 2:
                idx_fatura = j
            if v and "desconto" in str(v).lower() and idx_fatura and idx_fatura < j < idx_fatura + 3:
                idx_desconto = j
            if v and "custo" in str(v).lower() and idx_fatura and idx_fatura < j < idx_fatura + 4:
                idx_custo = j

    data_start = (header2_row or header1_row) + 1
    registros  = []
    for row in rows[data_start:]:
        mat  = row[idx_mat]      if idx_mat      is not None else None
        nome = row[idx_nome]     if idx_nome     is not None else None
        fat  = row[idx_fatura]   if idx_fatura   is not None else None
        desc = row[idx_desconto] if idx_desconto is not None else None
        custo = row[idx_custo]   if idx_custo    is not None else None
        if mat and isinstance(mat, (int, float)):
            registros.append({
                "matricula":         int(mat),
                "nome":              nome,
                "desconto_esperado": float(desc)  if isinstance(desc,  (int, float)) else 0.0,
                "fatura_esperada":   float(fat)   if isinstance(fat,   (int, float)) else 0.0,
                "custo_esperado":    float(custo) if isinstance(custo, (int, float)) else 0.0,
            })
    return pd.DataFrame(registros)


def conciliar(df_fatura, df_referencia):
    df = df_referencia.merge(df_fatura, on="matricula", how="outer")
    df["desconto_fatura"]   = df["desconto_fatura"].fillna(0.0)
    df["desconto_esperado"] = df["desconto_esperado"].fillna(0.0)
    df["diferenca"]         = (df["desconto_fatura"] - df["desconto_esperado"]).round(4)

    def status(row):
        if abs(row["diferenca"]) <= TOLERANCIA: return "✅ OK"
        elif row["desconto_esperado"] == 0:     return "👻 Só na fatura"
        elif row["desconto_fatura"]   == 0:     return "🔍 Só na referência"
        else:                                   return "💰 Divergência de valor"

    df["status"] = df.apply(status, axis=1)
    return df


# ════════════════════════════════════════════════════════════════════════
# REGISTRO DE PARSERS
# Chave: substring(s) presentes no nome do arquivo (lowercase)
# Valor: função de parse correspondente
# ════════════════════════════════════════════════════════════════════════

PARSERS = {
    ("bradesco", "dental"): parse_fatura_bradesco_dental,
    # Exemplo para adicionar novo fornecedor:
    # ("unimed",):  parse_fatura_unimed,
    # ("uniodonto",): parse_fatura_uniodonto,
}


def identificar_fornecedor(nome_arquivo):
    """Retorna (chave, função_parser) para o arquivo, ou (None, None) se não reconhecido."""
    nome_lower = nome_arquivo.lower()
    for chave, fn in PARSERS.items():
        if all(k in nome_lower for k in chave):
            return chave, fn
    return None, None


REFERENCIA_KEYWORDS = ("descontos", "proventos")

print("✅ Parsers carregados.")
print(f"   Fornecedores suportados: {len(PARSERS)}")

In [ ]:
#@title Célula 3 · Seleção de arquivos { display-mode: "form" }
#@markdown Faça o upload dos arquivos um a um. Clique em **Confirmar** quando terminar.

from IPython.display import clear_output

# ── Widgets ──────────────────────────────────────────────────────────────────
btn_upload    = widgets.Button(description="📂 Selecionar arquivo", button_style="info",
                               layout=widgets.Layout(width="200px", height="36px"))
btn_confirmar = widgets.Button(description="✔ Confirmar e processar", button_style="success",
                               layout=widgets.Layout(width="220px", height="36px"),
                               disabled=True)
btn_limpar    = widgets.Button(description="✖ Limpar seleção", button_style="warning",
                               layout=widgets.Layout(width="160px", height="36px"))
out_status    = widgets.Output()

def _render_status():
    with out_status:
        clear_output(wait=True)
        uploaded = estado["uploaded"]
        if not uploaded:
            print("Nenhum arquivo selecionado.")
            return

        linhas = []
        fatura_ok = referencia_ok = False

        for nome in uploaded:
            nome_lower = nome.lower()
            if all(k in nome_lower for k in REFERENCIA_KEYWORDS):
                papel = "📋 Referência interna"
                estado["arquivo_referencia"] = nome
                referencia_ok = True
            else:
                chave, _ = identificar_fornecedor(nome)
                if chave:
                    papel = f"🧾 Fatura — {' '.join(chave).title()}"
                    estado["arquivo_fatura"] = nome
                    estado["fornecedor"]     = chave
                    fatura_ok = True
                else:
                    papel = "⚠️  Não reconhecido"
            linhas.append(f"  {papel}: {nome}")

        for l in linhas:
            print(l)

        if fatura_ok and referencia_ok:
            print("\n✅ Arquivos prontos. Clique em Confirmar para processar.")
            btn_confirmar.disabled = False
        else:
            faltando = []
            if not fatura_ok:     faltando.append("fatura do fornecedor")
            if not referencia_ok: faltando.append("referência interna")
            print(f"\n⏳ Aguardando: {', '.join(faltando)}.")
            btn_confirmar.disabled = True

def on_upload(b):
    novos = files.upload()
    estado["uploaded"].update(novos)
    _render_status()

def on_limpar(b):
    estado["uploaded"].clear()
    estado["arquivo_fatura"] = estado["arquivo_referencia"] = estado["fornecedor"] = None
    btn_confirmar.disabled = True
    with out_status:
        clear_output(wait=True)
        print("Seleção limpa.")

btn_upload.on_click(on_upload)
btn_limpar.on_click(on_limpar)

# Confirmar apenas armazena sinal — processamento ocorre na Célula 4
def on_confirmar(b):
    with out_status:
        clear_output(wait=True)
        print("✅ Confirmado. Execute a Célula 4 para ver o resultado.")
    btn_confirmar.disabled = True

btn_confirmar.on_click(on_confirmar)

display(widgets.HBox([btn_upload, btn_limpar, btn_confirmar]))
display(out_status)
_render_status()

In [ ]:
#@title Célula 4 · Resultado { display-mode: "form" }
exportar_excel = False #@param {type:"boolean"}

from datetime import datetime as dt
from IPython.display import clear_output

# ── Validação ─────────────────────────────────────────────────────────────────
if not estado["arquivo_fatura"] or not estado["arquivo_referencia"]:
    display(HTML("""
    <div style="font-family:sans-serif; padding:12px; background:#fff3e0;
                border-left:4px solid #f57c00; border-radius:4px;">
      ⚠️ Nenhum arquivo confirmado. Execute a <strong>Célula 3</strong> primeiro.
    </div>
    """))
else:
    # ── Processamento ─────────────────────────────────────────────────────────
    _, fn_parser = identificar_fornecedor(estado["arquivo_fatura"])

    df_fatura     = fn_parser(estado["uploaded"][estado["arquivo_fatura"]])
    df_referencia = parse_referencia_interna(estado["uploaded"][estado["arquivo_referencia"]])
    df_result     = conciliar(df_fatura, df_referencia)

    fornecedor_label = " ".join(estado["fornecedor"]).title()

    # ── Métricas ──────────────────────────────────────────────────────────────
    total_ok          = (df_result["status"] == "✅ OK").sum()
    total_div         = (df_result["status"] != "✅ OK").sum()
    total_fatura_r    = df_result["desconto_fatura"].sum()
    total_ref_r       = df_result["desconto_esperado"].sum()
    diff_total        = total_fatura_r - total_ref_r
    cor_div           = "#c62828" if total_div > 0 else "#2e7d32"
    cor_diff          = "#c62828" if abs(diff_total) > TOLERANCIA else "#2e7d32"

    # ── Resumo ────────────────────────────────────────────────────────────────
    display(HTML(f"""
    <div style="font-family:sans-serif; padding:20px; background:#f8f9fa;
                border-left:4px solid #1565C0; border-radius:6px; margin-bottom:16px;">
      <h3 style="margin:0 0 14px 0; color:#1a1a2e; font-size:16px;">
        📊 Conciliação — {fornecedor_label}
        <span style="font-weight:normal; color:#555; font-size:13px; margin-left:8px;">
          {dt.today().strftime('%d/%m/%Y')}
        </span>
      </h3>
      <table style="border-collapse:collapse; width:100%; font-size:14px;">
        <tr>
          <td style="padding:5px 20px 5px 0; color:#555;">Colaboradores analisados</td>
          <td style="font-weight:600;">{len(df_result)}</td>
        </tr>
        <tr>
          <td style="padding:5px 20px 5px 0; color:#555;">✅ Sem divergência</td>
          <td style="font-weight:600; color:#2e7d32;">{total_ok}</td>
        </tr>
        <tr>
          <td style="padding:5px 20px 5px 0; color:#555;">⚠️ Com divergência</td>
          <td style="font-weight:600; color:{cor_div};">{total_div}</td>
        </tr>
        <tr><td colspan="2"><hr style="margin:10px 0; border:none; border-top:1px solid #ddd;"></td></tr>
        <tr>
          <td style="padding:5px 20px 5px 0; color:#555;">Total fatura (fornecedor)</td>
          <td style="font-weight:600;">R$ {total_fatura_r:.2f}</td>
        </tr>
        <tr>
          <td style="padding:5px 20px 5px 0; color:#555;">Total esperado (referência)</td>
          <td style="font-weight:600;">R$ {total_ref_r:.2f}</td>
        </tr>
        <tr>
          <td style="padding:5px 20px 5px 0; color:#555;">Diferença total</td>
          <td style="font-weight:700; color:{cor_diff};">R$ {diff_total:+.2f}</td>
        </tr>
      </table>
    </div>
    """))

    # ── Tabela detalhada ──────────────────────────────────────────────────────
    df_exibir = df_result[[
        "matricula", "nome", "qtd_vidas",
        "desconto_esperado", "desconto_fatura", "diferenca", "status"
    ]].copy()
    df_exibir.columns = [
        "Matrícula", "Nome", "Vidas",
        "Esperado (R$)", "Fatura (R$)", "Diferença (R$)", "Status"
    ]

    def _cor_linha(row):
        base = "background-color:{c}; color:#222; font-size:13px;"
        c = "#e8f5e9" if "OK" in row["Status"] else "#fff8e1"
        return [base.format(c=c)] * len(row)

    display(
        df_exibir.style
        .apply(_cor_linha, axis=1)
        .format({"Esperado (R$)": "{:.2f}", "Fatura (R$)": "{:.2f}", "Diferença (R$)": "{:+.2f}"})
        .set_table_styles([{
            "selector": "th",
            "props": [("background-color", "#1565C0"), ("color", "white"),
                      ("font-size", "13px"), ("padding", "8px 12px"),
                      ("text-align", "left")]
        }])
        .set_properties(**{"padding": "6px 12px"})
        .hide(axis="index")
    )

    # ── Export opcional ───────────────────────────────────────────────────────
    if exportar_excel:
        nome_export = f"conciliacao_{'_'.join(estado['fornecedor'])}_{dt.today().strftime('%m%Y')}.xlsx"
        df_result.to_excel(nome_export, index=False)
        files.download(nome_export)
        display(HTML(f"<p style='font-family:sans-serif; font-size:13px; color:#1565C0;'>📥 Exportado: <strong>{nome_export}</strong></p>"))
